# 11 — Inference Learning: Updating `D` Across Episodes

`AgentRuntime` runs an Active Inference perception-action loop over the A/B/C/D matrices produced by COGANT's reverse pipeline. This notebook walks through **multi-episode learning** — specifically, how the prior `D` over hidden states should be updated so the agent's starting belief tracks the distribution it actually sees across episodes.

We take the 'simple state' zoo example as our source, synthesise a COGANT package from it, run five episodes, and plot the VFE trajectory together with the evolving `D` prior.

**Run from the `cogant/` directory:**
```bash
uv run jupyter nbconvert --to notebook --execute docs/notebooks/11_inference_learning.ipynb
```

## 1. Source: `examples/zoo/01_simple_state/`

`01_simple_state` is a one-file zoo module containing a `BeliefState` class with `update_state` and `get_state` methods — enough surface area for COGANT to produce a non-trivial A/B/C/D.

In [1]:
from __future__ import annotations

import math
import tempfile
from pathlib import Path

from cogant.api.session import Session

FIXTURE = Path("examples/zoo/01_simple_state/").resolve()
if not FIXTURE.exists():
    raise FileNotFoundError(f"Zoo fixture not found at {FIXTURE}")

workdir = Path(tempfile.mkdtemp(prefix="cogant_nb11_"))
session = Session(repo_path=FIXTURE)
session.export_all(str(workdir))
print(f"Exported to: {workdir}")
print("Artifacts:")
for key, path in sorted(session.export_artifacts.items()):
    print(f"  {key:<20} {path}")

Exported to: /var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb11_mtdkti6a
Artifacts:
  gnn_package          /var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb11_mtdkti6a/gnn_package
  process_model        /var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb11_mtdkti6a/process_model.json
  program_graph        /var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb11_mtdkti6a/program_graph.json
  state_space          /var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb11_mtdkti6a/state_space.json
  syntax_tree          /var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb11_mtdkti6a/syntax_tree.json


## 2. Parse the emitted GNN model

`export_all` writes a `model.gnn.md` inside a synthesised package directory. We feed it to `parse_gnn` to recover A/B/C/D as plain nested lists. If COGANT did not emit a complete model for this tiny fixture, we fall back to a small hand-constructed one so the rest of the notebook still runs.

In [2]:
from cogant.reverse import parse_gnn

gnn_candidates = sorted(workdir.rglob("model.gnn.md"))
A: list[list[float]]
B: list[list[list[float]]]
C: list[float]
D: list[float]

if gnn_candidates:
    model = parse_gnn(gnn_candidates[0])
    print(f"Parsed {gnn_candidates[0]}")
    print(f"  hidden_states = {model.hidden_states}")
    print(f"  observations  = {model.observations}")
    print(f"  actions       = {model.actions}")
    A = [list(row) for row in (model.A or [])]
    B = [[list(r) for r in plane] for plane in (model.B or [])]
    C = list(model.C or [])
    D = list(model.D or [])
else:
    A = B = C = D = None  # type: ignore[assignment]

need_fallback = not (A and B and C and D) or len(D) < 2
if need_fallback:
    print("\nZoo model too small for a learning demo — using fallback 4-state / 4-obs / 2-action model.")
    n_s, n_o, n_a = 4, 4, 2
    # Diagonal-ish likelihood so observations track states.
    A = [[0.9 if i == j else 0.1 / (n_s - 1) for j in range(n_s)] for i in range(n_o)]
    # Two actions: 'stay' (identity) and 'shift' (cyclic).
    stay = [[1.0 if i == j else 0.0 for j in range(n_s)] for i in range(n_s)]
    shift = [[1.0 if i == (j + 1) % n_s else 0.0 for j in range(n_s)] for i in range(n_s)]
    # B layout: B[i][j][a]
    B = [[[stay[i][j], shift[i][j]] for j in range(n_s)] for i in range(n_s)]
    C = [0.0] * n_o
    C[0] = 1.0  # prefer observation 0
    D = [1.0 / n_s] * n_s

print(f"\nA shape: {len(A)} x {len(A[0]) if A else 0}")
print(f"B shape: {len(B)} x {len(B[0]) if B else 0} x {len(B[0][0]) if B and B[0] else 0}")
print(f"C     : {C}")
print(f"D     : {D}")

Parsed /var/folders/vc/rgmbpjpj0dbg61vr54xjskc80000gn/T/cogant_nb11_mtdkti6a/gnn_package/model.gnn.md
  hidden_states = ['s_f0']
  observations  = ['o_m0']
  actions       = ['u_c0', 'u_c1']

Zoo model too small for a learning demo — using fallback 4-state / 4-obs / 2-action model.

A shape: 4 x 4
B shape: 4 x 4 x 2
C     : [1.0, 0.0, 0.0, 0.0]
D     : [0.25, 0.25, 0.25, 0.25]


## 3. Create an `AgentRuntime`

`AgentRuntime.from_matrices_dict` builds a runtime from a plain dict with keys `A`, `B`, `C`, `D` and attaches sensible default helpers for likelihood, transition, and preference scoring.

In [3]:
from cogant.runtime import AgentRuntime

runtime = AgentRuntime.from_matrices_dict({"A": A, "B": B, "C": C, "D": D})
print(f"n_states  = {runtime._n_states}")  # noqa: SLF001
print(f"n_obs     = {runtime._n_obs}")      # noqa: SLF001
print(f"n_actions = {runtime._n_actions}")  # noqa: SLF001
print(f"D (prior) = {[round(v, 4) for v in runtime.D]}")

n_states  = 4
n_obs     = 4
n_actions = 2
D (prior) = [0.25, 0.25, 0.25, 0.25]


## 4. Run five episodes and update `D`

Each episode:

1. Starts from the current `D` prior.
2. Runs `run_n_steps(steps_per_episode)` — a perception-action loop that updates the belief step by step.
3. Records the **per-step VFE** so we can plot a learning curve.
4. Updates `D` to the running mean of the final posteriors seen so far. That's a Laplace-style online prior update — simple, numerically stable, and consistent with a uniform pseudocount prior.

This is a deliberately minimal learning loop assembled from the public runtime API. It's *not* the only way to learn `D`, but it shows the mechanics in a single cell.

In [4]:
STEPS_PER_EPISODE = 12
N_EPISODES = 5

D_trajectory: list[list[float]] = [list(runtime.D)]
vfe_trajectory: list[float] = []
per_step_vfes: list[list[float]] = []
final_posteriors: list[list[float]] = []

running_sum = [0.0] * runtime._n_states  # noqa: SLF001

for episode in range(N_EPISODES):
    steps = runtime.run_n_steps(STEPS_PER_EPISODE, initial_state=list(runtime.D))
    step_vfes = [s.free_energy for s in steps]
    per_step_vfes.append(step_vfes)
    mean_vfe = sum(step_vfes) / len(step_vfes)
    vfe_trajectory.append(mean_vfe)

    final_post = list(steps[-1].state_dist)
    final_posteriors.append(final_post)

    # Laplace-style running-mean update of D.
    running_sum = [a + b for a, b in zip(running_sum, final_post)]
    new_D = [v / (episode + 1) for v in running_sum]
    # Re-normalise to protect against floating-point drift.
    total = sum(new_D) or 1.0
    runtime.D = [v / total for v in new_D]
    D_trajectory.append(list(runtime.D))

    print(
        f"episode {episode + 1}: mean_vfe={mean_vfe:+.4f}  "
        f"final_post={[round(v, 3) for v in final_post]}  "
        f"D={[round(v, 3) for v in runtime.D]}"
    )

episode 1: mean_vfe=+1.4615  final_post=[1.0, 0.0, 0.0, 0.0]  D=[1.0, 0.0, 0.0, 0.0]
episode 2: mean_vfe=+0.1054  final_post=[1.0, 0.0, 0.0, 0.0]  D=[1.0, 0.0, 0.0, 0.0]
episode 3: mean_vfe=+0.1054  final_post=[1.0, 0.0, 0.0, 0.0]  D=[1.0, 0.0, 0.0, 0.0]
episode 4: mean_vfe=+0.1054  final_post=[1.0, 0.0, 0.0, 0.0]  D=[1.0, 0.0, 0.0, 0.0]
episode 5: mean_vfe=+0.1054  final_post=[1.0, 0.0, 0.0, 0.0]  D=[1.0, 0.0, 0.0, 0.0]


## 5. VFE trajectory across episodes

Plot the mean variational free energy per episode. A decreasing trajectory is the expected signature of learning — the agent's prior is getting closer to the distribution it actually sees, so its belief updates become cheaper per step.

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: mean VFE per episode.
ax1.plot(range(1, N_EPISODES + 1), vfe_trajectory, marker="o")
ax1.set_xlabel("episode")
ax1.set_ylabel("mean VFE")
ax1.set_title("Mean VFE per episode")
ax1.grid(True, alpha=0.3)

# Right: D prior as a stacked bar across episodes.
n_s = len(D_trajectory[0])
episodes = list(range(len(D_trajectory)))
bottom = [0.0] * len(D_trajectory)
for s in range(n_s):
    heights = [d[s] for d in D_trajectory]
    ax2.bar(episodes, heights, bottom=bottom, label=f"state {s}")
    bottom = [b + h for b, h in zip(bottom, heights)]
ax2.set_xlabel("episode (0 = initial)")
ax2.set_ylabel("D prior mass")
ax2.set_title("Prior D across episodes")
ax2.legend(loc="upper right", fontsize=8)

plt.tight_layout()
fig

<Figure size 1000x400 with 2 Axes>

## 6. Sanity checks

- Every `D` snapshot should sum to ~1 (normalised distribution).
- The length of `vfe_trajectory` should equal `N_EPISODES`.
- VFE values should be finite — no NaN, no infinities.

In [6]:
for i, d in enumerate(D_trajectory):
    total = sum(d)
    assert abs(total - 1.0) < 1e-6, f"D[{i}] sums to {total}"

assert len(vfe_trajectory) == N_EPISODES
for i, v in enumerate(vfe_trajectory):
    assert math.isfinite(v), f"vfe_trajectory[{i}] = {v}"

print(f"OK — {N_EPISODES} episodes, {len(D_trajectory)} D snapshots, all finite.")

OK — 5 episodes, 6 D snapshots, all finite.


## 7. Takeaways

- `AgentRuntime.from_matrices_dict` is the fastest way to spin up a runtime from raw A/B/C/D values — no need to synthesise a package first.
- Multi-episode learning is just 'run an episode, update a prior, repeat'; the runtime gives you per-step VFE for free.
- The update rule shown here (Laplace-style running mean of final posteriors) is a baseline; future work in `cogant.runtime` will expose richer `EpisodeResult` / `MultiEpisodeResult` helpers — watch `runtime/loop.py` for the API.
- Plot the VFE trajectory *and* the `D` bars together: a decreasing VFE without movement in `D` usually means learning is happening elsewhere (e.g. the `A` likelihood).